# Unidad 2: Aprendizaje Automático
## 🔄 Validación Cruzada — K-Fold Cross-Validation
### Inteligencia Artificial — Lic. en Sistemas — FCAD/UNER

---
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/20_KFold.ipynb)


## 🎯 ¿Qué vamos a aprender?

Hasta ahora evaluamos nuestros modelos con un único split train/test. Pero ese resultado depende de **cómo fue la división**: con otro `random_state` podríamos obtener resultados muy diferentes. La **Validación Cruzada K-Fold** soluciona este problema.

Al finalizar, vas a poder:
- ✅ Entender los problemas del enfoque simple train/test
- ✅ Implementar **K-Fold Cross-Validation** manualmente con `KFold`
- ✅ Promediar métricas sobre los K folds para una estimación más robusta
- ✅ Visualizar la variabilidad entre folds
- ✅ Saber cuándo y cómo usar el **modelo final** entrenado con todos los datos

---

## 🧠 Marco Teórico

### ❓ ¿Por qué no alcanza un solo split train/test?

Un único split tiene dos problemas:

1. **Alta varianza**: el score depende de qué datos caen en train y cuáles en test.
2. **Sub-uso de datos**: con 80/20, el 20% de los datos nunca se usa para entrenar.

### 🔄 ¿Cómo funciona K-Fold?

```
Dataset completo:  [─────────────────────────────────────────]

k=5 Folds:
  Fold 1:  [TEST│──────TRAIN──────────────────────────]
  Fold 2:  [────│TEST │──────TRAIN────────────────────]
  Fold 3:  [────────────│TEST │──────TRAIN────────────]
  Fold 4:  [────────────────────│TEST │───────TRAIN───]
  Fold 5:  [────────────────────────────│TEST │──TRAIN]

 → 5 scores → promediar → estimación robusta
```

**Ventajas:**
- Cada muestra es usada exactamente **1 vez como test** y **K−1 veces como train**
- El score promedio es mucho menos sensible a la partición aleatoria
- La desviación estándar entre folds indica la **variabilidad** del modelo

**Parámetros clave:**

| Parámetro | Descripción | Valor típico |
|-----------|-------------|-------------|
| `n_splits` | Número de folds (K) | 5 o 10 |
| `shuffle` | Mezclar el dataset antes de dividir | `True` |
| `random_state` | Semilla para reproducibilidad | Cualquier entero |

### 🏁 ¿Qué hacemos después? El Modelo Final

K-Fold nos da una **estimación del rendimiento**, pero **no el modelo en producción**. Una vez que sabemos que el modelo es bueno, lo entrenamos con **todo el dataset** para aprovechar todos los datos disponibles.

> 📌 **Referencias:**
> - Scikit-learn: [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
> - Scikit-learn: [Cross-validation: evaluating estimator performance](https://scikit-learn.org/stable/modules/cross_validation.html)
> - Géron, A. (2019). *Hands-On Machine Learning*, Cap. 2. O'Reilly.
> - Kohavi, R. (1995). [A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection](https://dl.acm.org/doi/10.5555/1643031.1643047). *IJCAI*.

---

## 📦 Paso 1: Importar las Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression

# 🔄 La clase KFold para validación cruzada manual
from sklearn.model_selection import KFold

print('✅ Librerías importadas correctamente!')

## 📂 Paso 2: Cargar el Dataset

In [ ]:
# 📥 Cargar dataset Titanic
df = pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/titanic.csv')
df['male'] = df['Sex'] == 'male'

X = df[['Pclass', 'male', 'Age', 'Siblings/Spouses', 'Parents/Children', 'Fare']].values
y = df['Survived'].values

print(f'📐 Dataset: {X.shape[0]} pasajeros × {X.shape[1]} features')
print(f'📊 Distribución: {(y==1).sum()} sobrevivieron ({(y==1).mean()*100:.1f}%), '
      f'{(y==0).sum()} no ({(y==0).mean()*100:.1f}%)')

## 🔄 Paso 3: Implementar K-Fold Manualmente

Usamos `KFold` para generar los índices de los folds y entrenamos el modelo en cada iteración.

In [ ]:
K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)

scores = []
print(f'🔄 Ejecutando {K}-Fold Cross-Validation...')
print(f'{"Fold":>5} {"Train":>7} {"Test":>7} {"Accuracy":>10}')
print('-' * 35)

for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    scores.append(acc)
    print(f'{fold:>5} {len(train_idx):>7} {len(test_idx):>7} {acc:>10.4f}')

print('-' * 35)
print(f'\n📊 Resultados:')
print(f'  Accuracy por fold:   {[f"{s:.4f}" for s in scores]}')
print(f'  Media (μ):           {np.mean(scores):.4f}')
print(f'  Desviación est. (σ): {np.std(scores):.4f}')
print(f'  Rango:               [{min(scores):.4f}, {max(scores):.4f}]')

## 📊 Paso 4: Visualizar los Scores por Fold

In [ ]:
mean_score = np.mean(scores)
std_score  = np.std(scores)

fig, ax = plt.subplots(figsize=(7, 4))

colores = ['tomato' if abs(s - mean_score) > std_score else 'steelblue' for s in scores]
bars = ax.bar([f'Fold {i+1}' for i in range(K)], scores,
              color=colores, edgecolor='black', alpha=0.85)

# Línea de media
ax.axhline(y=mean_score, color='forestgreen', linestyle='--', linewidth=2,
           label=f'Media: {mean_score:.4f}')
# Banda ±σ
ax.axhspan(mean_score - std_score, mean_score + std_score, alpha=0.1,
           color='green', label=f'±1σ: {std_score:.4f}')

for bar, val in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.003,
            f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

ax.set_ylim(max(0, min(scores) - 0.05), min(1.05, max(scores) + 0.05))
ax.set_ylabel('Accuracy')
ax.set_title(f'🔄 Accuracy por Fold — {K}-Fold CV (Logistic Regression)', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f'\n💡 Los folds en rojo están a más de 1σ de la media.')

## 🔬 Paso 5: Comparar K-Fold con Split Único

Comparamos la estimación de K-Fold con varios splits aleatorios simples para ver la variabilidad.

In [ ]:
from sklearn.model_selection import train_test_split

# 20 splits aleatorios distintos
scores_split = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=seed)
    m = LogisticRegression(max_iter=1000)
    m.fit(Xtr, ytr)
    scores_split.append(m.score(Xte, yte))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(20), scores_split, 'o-', color='steelblue', linewidth=1.5,
        markersize=6, label='Split único (20 semillas distintas)')
ax.axhline(y=mean_score, color='forestgreen', linestyle='--', linewidth=2,
           label=f'K-Fold CV media: {mean_score:.4f}')
ax.axhspan(mean_score - std_score, mean_score + std_score, alpha=0.15,
           color='green', label=f'K-Fold ±1σ')

ax.set_xlabel('Semilla (random_state)', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('🔬 Variabilidad del Split Único vs Estabilidad de K-Fold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nSplit único — Media: {np.mean(scores_split):.4f}, σ: {np.std(scores_split):.4f}')
print(f'K-Fold CV  — Media: {mean_score:.4f}, σ: {std_score:.4f}')
print('\n💡 El K-Fold da una estimación más estable y confiable.')

## 🏁 Paso 6: Entrenar el Modelo Final con Todos los Datos

Una vez validado el rendimiento con K-Fold, entrenamos el **modelo definitivo** con **todo el dataset** para maximizar la información utilizada.

In [ ]:
# 🏁 Modelo final: entrenado con TODOS los datos
modelo_final = LogisticRegression(max_iter=1000)
modelo_final.fit(X, y)   # ← Sin split: todos los datos para entrenar

print('=' * 55)
print('🏁 MODELO FINAL ENTRENADO CON TODO EL DATASET')
print('=' * 55)
print(f'  Muestras de entrenamiento: {len(X)}')
print(f'  Rendimiento estimado (K-Fold CV): {mean_score:.4f} ± {std_score:.4f}')
print()
print('  ⚠️  No podemos calcular accuracy "real" sobre el')
print('     conjunto de entrenamiento completo: sería optimista.')
print('     El K-Fold ya nos dio la mejor estimación disponible.')
print()
print('  Coeficientes del modelo final:')
features = ['Pclass', 'male', 'Age', 'Siblings/Spouses', 'Parents/Children', 'Fare']
for feat, coef in zip(features, modelo_final.coef_[0]):
    signo = '⬆️' if coef > 0 else '⬇️'
    print(f'    {feat:<22}: {coef:>+7.4f} {signo}')

## 🏁 Conclusiones

En este notebook aprendimos:

1. ❓ Un único split train/test tiene **alta varianza**: el score cambia con cada `random_state`.
2. 🔄 **K-Fold CV** divide el dataset en K partes, entrena K modelos y promedia los resultados para una estimación más robusta.
3. 📊 La **desviación estándar** entre folds indica cuán estable es el modelo: σ pequeño = modelo consistente.
4. 🏁 Después de validar con K-Fold, el **modelo final** se re-entrena con **TODOS** los datos.
5. 💡 Típicamente usamos K=5 o K=10 como valor estándar.

### ➡️ Próximo notebook: Comparar múltiples modelos con K-Fold

---

## 📚 Referencias

- Kohavi, R. (1995). [A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection](https://dl.acm.org/doi/10.5555/1643031.1643047). *IJCAI*.
- Scikit-learn: [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- Scikit-learn: [Cross-validation User Guide](https://scikit-learn.org/stable/modules/cross_validation.html)
- Géron, A. (2019). *Hands-On Machine Learning*, Cap. 2. O'Reilly.